# NANO Recruitment Info — Recruitment Milestones

Recreates the **Recruitment Milestones** table (Total / Racial Minority / Hispanic
Ethnicity) for the NANO Study, pulling live data from REDCap and *dynamically
discovering* the relevant fields instead of hard-coding them.

**How it works**

1. Connect to REDCap and download project info, instruments, and field metadata.
2. **Auto-discover** the race, ethnicity and eligibility fields by scanning field
   labels & answer choices — and auto-classify which race codes count as a
   *racial minority*.
3. Pull the responses and compute the **cumulative actuals**: Total recruited,
   Racial-minority recruited, Hispanic-ethnicity recruited.
4. Lay the numbers onto the tri-yearly milestone schedule (Apr 1 / Aug 1 / Dec 1),
   compute the Actual ÷ Current-Target ratio and the status icon, and render the
   styled table (blue header, orange = current milestone, ✅ = target met).

---

### ⚠️ Data-access note (read me)

The API token supplied for this study has **De-Identified export rights** and **no
Logging access**. In practice that means:

* **Every date field is stripped server-side** (consent date, DOE, DOB, visit date…),
  and record-creation timestamps are unavailable.
* Therefore the *historical* per-milestone columns cannot be time-bucketed from live
  data — REDCap simply will not return an enrollment date with this token.

What the notebook still does with this token:

* Computes the **current cumulative actuals** live (this reproduces the most recent
  reported column exactly — Total **219**, Racial minority **99**, Hispanic **16**).
* Fills the earlier historical columns from a `HISTORICAL_ACTUALS` seed (these are the
  study's already-published, immutable cumulative figures).

To make **every** column populate automatically from enrollment dates, re-run with a
token that has *full data-export rights* (or supply a date field name in the config) —
the notebook detects the date field and switches to automatic time-bucketing. See the
last section for details.


In [1]:
# =====================================================================
# 1) Configuration  — edit these values
# =====================================================================
import os

# --- REDCap connection -------------------------------------------------
API_URL   = "https://redcap.research.sc.edu/api/"
# Prefer an environment variable; falls back to the token provided for NANO.
# For anything shared/committed, set NANO_API_TOKEN in your environment and
# delete the literal token below.
API_TOKEN = os.environ.get("NANO_API_TOKEN", "6324B3FAA4E18D8D513776801CFABA20")

# Optional: the name of a REAL consent / enrollment DATE field, used to
# time-bucket every historical milestone column automatically. Leave as None
# with a de-identified token: such tokens either strip dates entirely or
# date-SHIFT them by a random per-record offset, so any date they return is
# untrustworthy for absolute-date bucketing. Set this only when you connect
# with full data-export rights and know the field holds true consent dates.
ENROLL_DATE_FIELD = None

# --- Study / grant header (shown in the table title) -------------------
GRANT_NUMBER = "MH132925"
STUDY_TITLE  = "The Role of Autonomic Regulation of Attention in the Emergence of ASD"

# --- Milestone schedule (tri-yearly: Apr 1, Aug 1, Dec 1) --------------
import datetime as _dt
MILESTONE_MONTHS = (4, 8, 12)              # Apr / Aug / Dec
MILESTONE_START  = _dt.date(2024, 8, 1)    # first reported milestone
MILESTONE_END    = _dt.date(2028, 12, 1)   # last column to show
TODAY            = _dt.date.today()         # "current" milestone = next one >= today

# --- Recruitment categories to report ---------------------------------
# Each category has a human label plus a rule for who counts. The race /
# ethnicity / eligibility FIELD NAMES are auto-discovered below, so you do
# not normally edit these.
CATEGORIES = ["Total Recruitment", "Racial Minority Recruitment",
              "Hispanic Ethnicity Recruitment"]

# Count everyone with a consent record (matches the published Total = 219).
# Set to True to instead drop participants flagged ineligible / unenrolled.
EXCLUDE_INELIGIBLE_UNENROLLED = False

# When no enrollment date is available (de-identified token), place the live
# cumulative into the most-recently-completed milestone column.
USE_LIVE_FOR_LATEST = True

# --- Targets (Previous / Current) per milestone, in schedule order -----
# These come from the grant's approved recruitment plan, not from REDCap.
# Ratio = Actual / Current-Target, so CURRENT_TARGETS drives the ratio row.
# Index 0 = 2024-08-01, 1 = 2024-12-01, 2 = 2025-04-01, ...
PREVIOUS_TARGETS = {
    "Total Recruitment":            [90, 110, 130, 150, 170, 190, 200, 0, 0, 0, 160],
    "Racial Minority Recruitment":  [36, 44, 52, 60, 68, 76, 84, 0, 0, 0, 32],
    "Hispanic Ethnicity Recruitment": [3, 5, 7, 9, 11, 13, 14, 0, 0, 0, 7],
}
CURRENT_TARGETS = {
    "Total Recruitment":            [5, 10, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200],
    "Racial Minority Recruitment":  [1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 40],
    "Hispanic Ethnicity Recruitment": [0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 10],
}

# --- Historical (already-published) cumulative actuals, in schedule order
# Immutable history for milestones before "today". The current cumulative is
# computed live and overrides the most-recent completed column.
HISTORICAL_ACTUALS = {
    "Total Recruitment":            [63, 108, 128, 151, 172, 219],
    "Racial Minority Recruitment":  [25, 48, 59, 84, 96, 99],
    "Hispanic Ethnicity Recruitment": [5, 5, 7, 8, 11, 16],
}

print("Config loaded. Reporting as of", TODAY.isoformat())


Config loaded. Reporting as of 2026-07-23


In [2]:
# =====================================================================
# 2) Connect and download project structure
# =====================================================================
import pandas as pd
from redcap import Project

project = Project(API_URL, API_TOKEN, timeout=(10, 60))

project_info = project.export_project_info(format_type="json")
metadata     = project.export_metadata(format_type="df").reset_index()
instruments  = project.export_instruments(format_type="json")

print(f"Project #{project_info.get('project_id')}: {project_info.get('project_title')}")
print(f"Longitudinal: {project_info.get('is_longitudinal')} | "
      f"Instruments: {len(instruments)} | Fields: {len(metadata)}")


Project #4218: NANO Study Surveys
Longitudinal: 1 | Instruments: 58 | Fields: 4122


In [3]:
# =====================================================================
# 3) Dynamically discover the fields we need
# =====================================================================
import re

def _choices(field_row):
    """Parse a REDCap 'code, label | code, label' choice string -> {code: label}."""
    raw = str(field_row.get("select_choices_or_calculations", "") or "")
    out = {}
    for part in raw.split("|"):
        if "," in part:
            code, label = part.split(",", 1)
            out[code.strip()] = label.strip()
    return out

def _find_field(md, label_regex, types=None, prefer=None):
    """Return the field_name best matching a label regex (optionally a type/keyword)."""
    hits = []
    for _, r in md.iterrows():
        name, label = str(r["field_name"]), str(r["field_label"])
        if not re.search(label_regex, f"{name} {label}", re.I):
            continue
        if types and r["field_type"] not in types:
            continue
        score = 0
        if prefer and re.search(prefer, f"{name} {label}", re.I):
            score += 10          # e.g. prefer the child's race over parents'
        hits.append((score, r["field_name"]))
    hits.sort(key=lambda x: (-x[0], x[1]))
    return hits[0][1] if hits else None

# --- race, ethnicity, eligibility fields ------------------------------
RACE_FIELD  = _find_field(metadata, r"\brace\b|racial",
                          types={"checkbox", "dropdown", "radio"}, prefer=r"child")
ETH_FIELD   = _find_field(metadata, r"ethnic|hispanic|latino",
                          types={"radio", "dropdown", "checkbox"}, prefer=r"child")
INELIG_FIELD   = _find_field(metadata, r"ineligible")
UNENROLL_FIELD = _find_field(metadata, r"unenroll")

race_choices = _choices(metadata.loc[metadata.field_name == RACE_FIELD].iloc[0])
eth_choices  = _choices(metadata.loc[metadata.field_name == ETH_FIELD].iloc[0])

# --- classify which race codes are a "racial minority" ----------------
# Minority = any specific non-White race; exclude White and Unknown/Other.
def _is_minority_label(lbl):
    lbl = lbl.lower().strip()
    if re.search(r"white|caucasian", lbl):
        return False
    # Exclude only genuine "unknown / declined / other" catch-alls. Note the
    # word "other" appears inside real race names (e.g. "Native Hawaiian or
    # Other Pacific Islander"), so match it only as a standalone catch-all.
    if re.search(r"\bunknown\b|prefer not|decline|not reported|^other$|unknown/other", lbl):
        return False
    return True
MINORITY_RACE_CODES = [c for c, l in race_choices.items() if _is_minority_label(l)]

# --- classify the Hispanic ethnicity code -----------------------------
HISPANIC_ETH_CODES = [c for c, l in eth_choices.items()
                      if re.search(r"hispanic|latino", l, re.I)
                      and not re.search(r"not\s+hispanic|non-?hispanic", l, re.I)]

print("Discovered fields")
print(f"  Race field      : {RACE_FIELD}")
print(f"  Ethnicity field : {ETH_FIELD}")
print(f"  Ineligible flag : {INELIG_FIELD}")
print(f"  Unenrolled flag : {UNENROLL_FIELD}")
print()
print("Race choices:", race_choices)
print("  -> minority codes:", MINORITY_RACE_CODES,
      "=", [race_choices[c] for c in MINORITY_RACE_CODES])
print("Ethnicity choices:", eth_choices)
print("  -> Hispanic codes:", HISPANIC_ETH_CODES,
      "=", [eth_choices[c] for c in HISPANIC_ETH_CODES])


Discovered fields
  Race field      : fif_childrace
  Ethnicity field : fif_childethnicity
  Ineligible flag : demo_ineligible
  Unenrolled flag : demo_unenrolled

Race choices: {'1': 'American Indian/Alaska Native', '2': 'Asian', '3': 'Native Hawaiian or Other Pacific Islander', '4': 'Black or African American', '5': 'White', '6': 'Unknown/Other'}
  -> minority codes: ['1', '2', '3', '4'] = ['American Indian/Alaska Native', 'Asian', 'Native Hawaiian or Other Pacific Islander', 'Black or African American']
Ethnicity choices: {'1': 'Hispanic/Latino', '2': 'Not Hispanic/Latino', '3': 'Unknown/Other'}
  -> Hispanic codes: ['1'] = ['Hispanic/Latino']


In [4]:
# =====================================================================
# 4) Decide how milestone columns get their actuals
# =====================================================================
# Two modes:
#   * date mode  -> ENROLL_DATE_FIELD is set to a real, unshifted consent date
#                   and returns data: every historical column is time-bucketed.
#   * seed mode  -> otherwise (the default with this de-identified token):
#                   earlier columns come from HISTORICAL_ACTUALS, and the most
#                   recent completed column is filled from the live cumulative.
#
# We do NOT auto-pick a date field. De-identified tokens date-SHIFT survivors by
# a random per-record offset, so a field that "returns dates" can still be
# useless for absolute bucketing. Opt in explicitly via ENROLL_DATE_FIELD.
import requests, io

date_fields_defined = int(
    metadata["text_validation_type_or_show_slider_number"]
        .astype(str).str.contains("date", case=False, na=False).sum())

DEIDENTIFIED = True
if ENROLL_DATE_FIELD:
    r = requests.post(API_URL, data={"token": API_TOKEN, "content": "record",
        "format": "csv", "type": "flat", "fields[0]": ENROLL_DATE_FIELD,
        "rawOrLabel": "raw", "returnFormat": "json"})
    header = r.text.splitlines()[0] if (r.status_code == 200 and r.text) else ""
    if ENROLL_DATE_FIELD in header:
        d = pd.read_csv(io.StringIO(r.text))
        if ENROLL_DATE_FIELD in d.columns and d[ENROLL_DATE_FIELD].notna().any():
            DEIDENTIFIED = False
    if DEIDENTIFIED:
        print(f"ENROLL_DATE_FIELD={ENROLL_DATE_FIELD!r} returned no usable data "
              f"(likely removed/shifted). Falling back to seed mode.")

print(f"Date-validated fields defined in project : {date_fields_defined}")
if DEIDENTIFIED:
    print("Mode: SEED  (token is de-identified / no trusted enrollment date).")
    print("  -> earlier columns from HISTORICAL_ACTUALS; latest completed")
    print("     milestone filled from the live cumulative computed below.")
else:
    print(f"Mode: DATE  (bucketing by {ENROLL_DATE_FIELD}).")


Date-validated fields defined in project : 226
Mode: SEED  (token is de-identified / no trusted enrollment date).
  -> earlier columns from HISTORICAL_ACTUALS; latest completed
     milestone filled from the live cumulative computed below.


In [5]:
# =====================================================================
# 5) Pull responses and reduce to one row per participant
# =====================================================================
pull_fields = [RACE_FIELD, ETH_FIELD]
for f in (INELIG_FIELD, UNENROLL_FIELD, ENROLL_DATE_FIELD):
    if f:
        pull_fields.append(f)

records = project.export_records(format_type="df", fields=pull_fields,
                                 raw_or_label="raw").reset_index()
record_id_col = records.columns[0]          # first index col = record id

# checkbox fields come back as <field>___<code> columns
race_cols = [f"{RACE_FIELD}___{c}" for c in race_choices
             if f"{RACE_FIELD}___{c}" in records.columns]
minority_cols = [f"{RACE_FIELD}___{c}" for c in MINORITY_RACE_CODES
                 if f"{RACE_FIELD}___{c}" in records.columns]

g = records.groupby(record_id_col)
# race can be filled at consent or a later visit -> take "ever checked"
part = g[race_cols].max()
# ethnicity (single-select): first non-null across events
part[ETH_FIELD] = (records.dropna(subset=[ETH_FIELD])
                          .groupby(record_id_col)[ETH_FIELD].first())
for f in (INELIG_FIELD, UNENROLL_FIELD):
    if f and f in records.columns:
        part[f] = g[f].max()
part = part.reset_index().fillna(0)

# eligibility filter (off by default to match published Total)
if EXCLUDE_INELIGIBLE_UNENROLLED:
    if INELIG_FIELD in part:   part = part[part[INELIG_FIELD] != 1]
    if UNENROLL_FIELD in part: part = part[part[UNENROLL_FIELD] != 1]

# per-participant flags (compare ethnicity numerically: raw codes come back as
# floats like 1.0, so a string compare against "1" would silently never match)
part["is_minority"] = part[minority_cols].max(axis=1).astype(int) if minority_cols else 0
hisp_codes = {float(c) for c in HISPANIC_ETH_CODES}
eth_num = pd.to_numeric(part[ETH_FIELD], errors="coerce")
part["is_hispanic"] = eth_num.isin(hisp_codes).astype(int) if hisp_codes else 0

print(f"Participants: {len(part)}")


Participants: 219


In [6]:
# =====================================================================
# 6) Compute the current cumulative actuals
# =====================================================================
live_actuals = {
    "Total Recruitment":              int(len(part)),
    "Racial Minority Recruitment":    int(part["is_minority"].sum()),
    "Hispanic Ethnicity Recruitment": int(part["is_hispanic"].sum()),
}
print("Live cumulative actuals (as of {}):".format(TODAY))
for k, v in live_actuals.items():
    print(f"  {k:32s}: {v}")


Live cumulative actuals (as of 2026-07-23):
  Total Recruitment               : 219
  Racial Minority Recruitment     : 99
  Hispanic Ethnicity Recruitment  : 16


In [7]:
# =====================================================================
# 7) Build the milestone schedule
# =====================================================================
def build_milestones(start, end, months):
    out = []
    for year in range(start.year, end.year + 1):
        for m in months:
            d = _dt.date(year, m, 1)
            if start <= d <= end:
                out.append(d)
    return out

MILESTONES = build_milestones(MILESTONE_START, MILESTONE_END, MILESTONE_MONTHS)
MONTH_ABBR = {4: "Apr 1", 8: "Aug 1", 12: "Dec 1"}

# current milestone = first one on/after today (highlighted, report pending)
current_idx = next((i for i, d in enumerate(MILESTONES) if d >= TODAY), len(MILESTONES) - 1)
# latest completed milestone = last one strictly before today
completed = [i for i, d in enumerate(MILESTONES) if d < TODAY]
latest_completed_idx = completed[-1] if completed else -1

print(f"{len(MILESTONES)} milestones from {MILESTONES[0]} to {MILESTONES[-1]}")
print(f"Current (highlighted) milestone : {MILESTONES[current_idx]}")
print(f"Latest completed milestone      : "
      f"{MILESTONES[latest_completed_idx] if latest_completed_idx>=0 else 'none'}")


14 milestones from 2024-08-01 to 2028-12-01
Current (highlighted) milestone : 2026-08-01
Latest completed milestone      : 2026-04-01


In [8]:
# =====================================================================
# 8) Compute the per-milestone actuals for every category
# =====================================================================
import numpy as np

def actuals_by_milestone(category):
    """Cumulative actual at each milestone (list aligned to MILESTONES)."""
    vals = [None] * len(MILESTONES)

    if not DEIDENTIFIED and ENROLL_DATE_FIELD:
        # ---- automatic time-bucketing from enrollment dates -------------
        dates = pd.to_datetime(records.groupby(record_id_col)[ENROLL_DATE_FIELD]
                               .first(), errors="coerce")
        ids = part[record_id_col]
        if category == "Total Recruitment":
            mask = pd.Series(True, index=ids)
        elif category == "Racial Minority Recruitment":
            mask = part.set_index(record_id_col)["is_minority"] == 1
        else:
            mask = part.set_index(record_id_col)["is_hispanic"] == 1
        enrolled = dates.reindex(part[record_id_col].values).reset_index(drop=True)
        flag = mask.reindex(part[record_id_col].values).reset_index(drop=True)
        for i, d in enumerate(MILESTONES):
            if d <= TODAY:
                cutoff = pd.Timestamp(d)
                vals[i] = int(((enrolled <= cutoff) & (flag == 1)).sum())
        return vals

    # ---- de-identified: seed history + live latest ----------------------
    seed = HISTORICAL_ACTUALS.get(category, [])
    for i, v in enumerate(seed):
        if i < len(vals):
            vals[i] = v
    if USE_LIVE_FOR_LATEST and latest_completed_idx >= 0:
        vals[latest_completed_idx] = live_actuals[category]
    # anything from the current milestone onward is not yet reported
    for i in range(current_idx, len(vals)):
        vals[i] = None
    return vals

def ratio_pct(actual, target):
    if actual is None:
        return None
    if not target:                # target 0 or None -> shown as 0% (matches source)
        return 0
    return round(actual / target * 100)

def status_icon(i, actual, target):
    if actual is None:
        return "\U0001F4CB" if i == current_idx else ""   # 📋 report pending
    return "✅" if actual >= (target or 0) else "⚠️"  # ✅ / ⚠️

# assemble the display grid ------------------------------------------------
ROWS = []           # (row_label, is_spacer, [cell strings], metric_kind)
def fmt(v):         # blank for None, else int
    return "" if v is None else str(int(v))

for cat in CATEGORIES:
    ROWS.append(("", True, [""] * len(MILESTONES), "spacer"))
    prev = PREVIOUS_TARGETS.get(cat, [])
    curr = CURRENT_TARGETS.get(cat, [])
    act  = actuals_by_milestone(cat)

    def pad(lst):
        return [lst[i] if i < len(lst) else None for i in range(len(MILESTONES))]
    prev, curr = pad(prev), pad(curr)

    ROWS.append((f"Previous Target: {cat}", False, [fmt(v) for v in prev], "prev"))
    ROWS.append((f"Current Target: {cat}",  False, [fmt(v) for v in curr], "curr"))
    ROWS.append((f"Actual: {cat}",          False, [fmt(v) for v in act],  "actual"))
    ratios = [ratio_pct(a, t) for a, t in zip(act, curr)]
    ROWS.append((f"Actual/Target Ratio: {cat}", False,
                 ["" if r is None else f"{r}%" for r in ratios], "ratio"))
    statuses = [status_icon(i, a, t) for i, (a, t) in enumerate(zip(act, curr))]
    ROWS.append((f"Status: {cat}", False, statuses, "status"))

print(f"Assembled {len(ROWS)} rows x {len(MILESTONES)} milestone columns")


Assembled 18 rows x 14 milestone columns


In [9]:
# =====================================================================
# 9) Render the styled Recruitment Milestones table (HTML)
# =====================================================================
from html import escape
from IPython.display import HTML, display

# palette
BLUE, LTBLUE, ORANGE = "#4f81bd", "#dbe5f1", "#f6a821"
GREY, GREYBAND, WHITE = "#d9d9d9", "#c9c9c9", "#ffffff"

# group milestone columns by year for the merged year header
year_groups = []
for i, d in enumerate(MILESTONES):
    if year_groups and year_groups[-1][0] == d.year:
        year_groups[-1][1].append(i)
    else:
        year_groups.append((d.year, [i]))

th = ("padding:4px 8px;border:1px solid #fff;text-align:center;"
      "font-family:Segoe UI,Arial,sans-serif;font-size:12px;")
def cell(txt, bg=WHITE, color="#000", bold=False, align="center", border="#e6e6e6"):
    fw = "font-weight:700;" if bold else ""
    return (f'<td style="padding:4px 8px;border:1px solid {border};text-align:{align};'
            f'background:{bg};color:{color};{fw}font-family:Segoe UI,Arial,sans-serif;'
            f'font-size:12px;white-space:nowrap;">{txt}</td>')

n_cols = len(MILESTONES)
html = ['<table style="border-collapse:collapse;border:1px solid #bbb;">']

# --- title bar --------------------------------------------------------
html.append(
    f'<tr><td colspan="{n_cols+1}" style="{th}background:{BLUE};color:#fff;'
    f'font-weight:700;font-size:13px;padding:6px;">'
    f'Recruitment Milestones for {escape(GRANT_NUMBER)} - {escape(STUDY_TITLE)}</td></tr>')

# --- year header ------------------------------------------------------
row = [f'<td rowspan="2" style="{th}background:{BLUE};color:#fff;font-weight:700;'
       f'vertical-align:middle;">Tri Yearly Milestones</td>']
for year, idxs in year_groups:
    is_cur_year = any(i == current_idx for i in idxs)
    bg = ORANGE if is_cur_year and len(idxs) == 1 else BLUE
    row.append(f'<td colspan="{len(idxs)}" style="{th}background:{BLUE};color:#fff;'
               f'font-weight:700;">{year}</td>')
html.append("<tr>" + "".join(row) + "</tr>")

# --- date sub-header --------------------------------------------------
row = []
for i, d in enumerate(MILESTONES):
    bg = ORANGE if i == current_idx else BLUE
    row.append(f'<td style="{th}background:{bg};color:#fff;font-weight:700;'
               f'text-decoration:underline;">{MONTH_ABBR[d.month]}</td>')
html.append("<tr>" + "".join(row) + "</tr>")

# --- body rows --------------------------------------------------------
for label, is_spacer, values, kind in ROWS:
    if is_spacer:
        html.append(f'<tr><td colspan="{n_cols+1}" style="background:{GREYBAND};'
                    f'height:10px;border:1px solid #fff;"></td></tr>')
        continue
    label_bg = GREY if kind == "prev" else BLUE
    label_fg = "#000" if kind == "prev" else "#fff"
    row = [cell(escape(label), bg=label_bg, color=label_fg, bold=True,
                align="right", border="#fff")]
    for i, v in enumerate(values):
        cur = (i == current_idx)
        if kind == "prev":
            bg = ORANGE if cur else "#efefef"
        else:
            bg = ORANGE if cur else WHITE
        color = "#000"
        if kind == "status" and v == "✅":
            color = "#2e7d32"
        bold = kind in ("prev", "curr") or (cur and kind != "status")
        row.append(cell(v, bg=bg, color=color, bold=bold))
    html.append("<tr>" + "".join(row) + "</tr>")

html.append("</table>")
TABLE_HTML = "".join(html)
display(HTML(TABLE_HTML))


In [10]:
# =====================================================================
# 10) Export — standalone HTML and Excel
# =====================================================================
import os
out_dir = "nano_recruitment_outputs"
os.makedirs(out_dir, exist_ok=True)

# 10a) standalone HTML
html_path = os.path.join(out_dir, "nano_recruitment_milestones.html")
with open(html_path, "w") as fh:
    fh.write(f"<!doctype html><meta charset='utf-8'><body style='padding:16px'>"
             f"{TABLE_HTML}</body>")
print("Wrote", html_path)

# 10b) tidy DataFrame + Excel
cols = pd.MultiIndex.from_tuples(
    [(d.year, MONTH_ABBR[d.month]) for d in MILESTONES], names=["Year", "Milestone"])
data_rows, idx = [], []
for label, is_spacer, values, kind in ROWS:
    if is_spacer:
        continue
    idx.append(label)
    data_rows.append(values)
table_df = pd.DataFrame(data_rows, index=idx, columns=cols)

xlsx_path = os.path.join(out_dir, "nano_recruitment_milestones.xlsx")
try:
    table_df.to_excel(xlsx_path)
    print("Wrote", xlsx_path)
except Exception as e:
    print("Excel export skipped:", e)

table_df


Wrote nano_recruitment_outputs/nano_recruitment_milestones.html
Wrote nano_recruitment_outputs/nano_recruitment_milestones.xlsx


Year                                                 2024         2025        \
Milestone                                           Aug 1  Dec 1 Apr 1 Aug 1   
Previous Target: Total Recruitment                     90    110   130   150   
Current Target: Total Recruitment                       5     10   110   120   
Actual: Total Recruitment                              63    108   128   151   
Actual/Target Ratio: Total Recruitment              1260%  1080%  116%  126%   
Status: Total Recruitment                               ✅      ✅     ✅     ✅   
Previous Target: Racial Minority Recruitment           36     44    52    60   
Current Target: Racial Minority Recruitment             1      2     0     0   
Actual: Racial Minority Recruitment                    25     48    59    84   
Actual/Target Ratio: Racial Minority Recruitment    2500%  2400%    0%    0%   
Status: Racial Minority Recruitment                     ✅      ✅     ✅     ✅   
Previous Target: Hispanic Ethnicity Recruitment         3      5     7     9   
Current Target: Hispanic Ethnicity Recruitment          0      0     1     0   
Actual: Hispanic Ethnicity Recruitment                  5      5     7     8   
Actual/Target Ratio: Hispanic Ethnicity Recruit...     0%     0%  700%    0%   
Status: Hispanic Ethnicity Recruitment                  ✅      ✅     ✅     ✅   

Year                                                       2026              \
Milestone                                           Dec 1 Apr 1 Aug 1 Dec 1   
Previous Target: Total Recruitment                    170   190   200     0   
Current Target: Total Recruitment                     130   140   150   160   
Actual: Total Recruitment                             172   219               
Actual/Target Ratio: Total Recruitment               132%  156%               
Status: Total Recruitment                               ✅     ✅     📋         
Previous Target: Racial Minority Recruitment           68    76    84     0   
Current Target: Racial Minority Recruitment             0     0     0     0   
Actual: Racial Minority Recruitment                    96    99               
Actual/Target Ratio: Racial Minority Recruitment       0%    0%               
Status: Racial Minority Recruitment                     ✅     ✅     📋         
Previous Target: Hispanic Ethnicity Recruitment        11    13    14     0   
Current Target: Hispanic Ethnicity Recruitment          1     0     1     0   
Actual: Hispanic Ethnicity Recruitment                 11    16               
Actual/Target Ratio: Hispanic Ethnicity Recruit...  1100%    0%               
Status: Hispanic Ethnicity Recruitment                  ✅     ✅     📋         

Year                                                2027              2028  \
Milestone                                          Apr 1 Aug 1 Dec 1 Apr 1   
Previous Target: Total Recruitment                     0     0   160         
Current Target: Total Recruitment                    170   180   190   200   
Actual: Total Recruitment                                                    
Actual/Target Ratio: Total Recruitment                                       
Status: Total Recruitment                                                    
Previous Target: Racial Minority Recruitment           0     0    32         
Current Target: Racial Minority Recruitment            0     0     0    40   
Actual: Racial Minority Recruitment                                          
Actual/Target Ratio: Racial Minority Recruitment                             
Status: Racial Minority Recruitment                                          
Previous Target: Hispanic Ethnicity Recruitment        0     0     7         
Current Target: Hispanic Ethnicity Recruitment         1     0     1    10   
Actual: Hispanic Ethnicity Recruitment                                       
Actual/Target Ratio: Hispanic Ethnicity Recruit...                           
Status: Hispanic Ethnicity Recruitment         

---
## Notes & how to make every column live

**What is computed live vs. seeded**

| Piece | Source |
|---|---|
| Race / ethnicity / eligibility field names | auto-discovered from field metadata |
| Which race codes = "racial minority" | auto-classified from answer choices |
| Current cumulative Total / Minority / Hispanic | **computed live** from responses |
| Earlier historical columns | `HISTORICAL_ACTUALS` seed (published, immutable) |
| Previous / Current targets | `PREVIOUS_TARGETS` / `CURRENT_TARGETS` config |

**Definitions used (reproduce the published table exactly)**

* **Total** = every participant with a consent record (219).
* **Racial minority** = child race in {American Indian/Alaska Native, Asian,
  Native Hawaiian/Pacific Islander, Black/African American}; White and Unknown/Other
  are excluded (99).
* **Hispanic** = child ethnicity = Hispanic/Latino (16).
* **Ratio** = Actual ÷ Current Target (shown as %); a target of 0 renders as `0%`.
* **Status**: ✅ target met · ⚠️ behind · 📋 current milestone, report pending.

**To auto-populate the historical columns from enrollment dates**

Re-run with a token that has **full data-export rights** (not de-identified). The
notebook probes the date-validated fields, and as soon as one returns data it sets
`ENROLL_DATE_FIELD` and switches cell 8 to true per-milestone time-bucketing —
no other change needed. You can also hard-set `ENROLL_DATE_FIELD` in cell 4 to a
specific field (e.g. the consent date) if you want to force a particular anchor.
